In [1]:
# =====================================================================
# CELL 0 — IMPORT, KONFIGURASI, MODEL, DAN PATH
# =====================================================================
from pathlib import Path
from typing import Any
from datetime import datetime
from itertools import combinations
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import wilcoxon

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, learning_curve, ShuffleSplit

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ---------------------------------------------------------------------
# Konfigurasi eksperimen
# ---------------------------------------------------------------------
waktu_sekarang = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Timestamp eksperimen saat ini: {waktu_sekarang}")

RANDOM_STATE = 42
N_SPLITS = 5
LABEL_COL = "label"
FILENAME_COL = "filename"

INPUT_DIR = Path("hasil_ekstraksi")

MODEL_OUT_DIR = Path("output/models")
GRAPH_OUT_DIR = Path("output/grafik")
METRIC_OUT_DIR = Path("output/metrik")

for _dir in (MODEL_OUT_DIR, GRAPH_OUT_DIR, METRIC_OUT_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

NON_FEATURE_COLS = [LABEL_COL, FILENAME_COL]

BASELINE_CSV = INPUT_DIR / "fitur_train_set.csv"

# SELURUH train set obfuskasi yang menghasilkan total 5.400 baris.
AUGMENTATION_FILES = [
    "fitur_train_set_obf_ASCII_Encoding.csv",
    "fitur_train_set_obf_String_Concatenation.csv",
    "fitur_train_set_obf_String_Reordering.csv",
    "fitur_train_set_obf_Token_Manipulation.csv",
]

EXPECTED_BASELINE_ROWS = 4949
EXPECTED_AUGMENTATION_ROWS = 5400

# Profil fold acuan dari hasil yang sudah Anda tetapkan.
# (jumlah validasi, malicious validasi, augmentasi aman, turunan validasi dihapus)
REFERENCE_FOLD_PROFILE = [
    (990, 301, 4340, 1060),
    (990, 301, 4296, 1104),
    (990, 301, 4292, 1108),
    (990, 302, 4348, 1052),
    (989, 301, 4324, 1076),
]

MODELS: dict[str, Any] = {
    "RandomForest": RandomForestClassifier(
        n_jobs=-1,
        random_state=RANDOM_STATE,
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
    ),
    "XGBoost": XGBClassifier(
        n_jobs=-1,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        verbosity=0,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
    ),
    "LightGBM": LGBMClassifier(
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=-1,
        max_depth=7,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        subsample_freq=1,
    ),
}

print("✅ [CELL 0] Import, direktori, dan konfigurasi model siap.")


Timestamp eksperimen saat ini: 20260809_101313
✅ [CELL 0] Import, direktori, dan konfigurasi model siap.


In [2]:
# =====================================================================
# CELL 1 — FUNGSI PEMBANTU, VALIDASI DATA, CV, DAN EKSPOR MODEL
# =====================================================================

def _get_feature_cols(df: pd.DataFrame) -> list[str]:
    """Mengambil kolom fitur dengan mengecualikan label dan filename."""
    return [c for c in df.columns if c not in NON_FEATURE_COLS]


def _validate_binary_labels(df: pd.DataFrame, source_name: str) -> None:
    """Memastikan label tersedia dan hanya berisi 0/1."""
    if LABEL_COL not in df.columns:
        raise KeyError(f"Kolom label '{LABEL_COL}' tidak ditemukan pada {source_name}.")
    labels = set(pd.Series(df[LABEL_COL]).dropna().unique().tolist())
    if not labels.issubset({0, 1}):
        raise ValueError(
            f"Label pada {source_name} harus biner 0/1, tetapi ditemukan: {sorted(labels)}"
        )


def _validate_required_columns(df: pd.DataFrame, source_name: str) -> None:
    """Memastikan kolom identitas minimum tersedia."""
    missing = [c for c in (LABEL_COL, FILENAME_COL) if c not in df.columns]
    if missing:
        raise KeyError(f"{source_name} tidak memiliki kolom wajib: {missing}")
    _validate_binary_labels(df, source_name)


def _evaluate_metrics(
    y_true: pd.Series | np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray | None = None,
) -> dict[str, float]:
    """Menghitung metrik klasifikasi biner secara aman."""
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)

    # labels=[0,1] mencegah warning confusion matrix saat salah satu kelas tidak muncul.
    tn, fp, fn, tp = confusion_matrix(
        y_true_arr, y_pred_arr, labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    result = {
        "accuracy": accuracy_score(y_true_arr, y_pred_arr),
        "precision": precision_score(y_true_arr, y_pred_arr, zero_division=0),
        "recall": recall_score(y_true_arr, y_pred_arr, zero_division=0),
        "f1": f1_score(y_true_arr, y_pred_arr, zero_division=0),
        "fpr": fpr,
        "fnr": fnr,
    }

    # ROC-AUC hanya valid bila y_true mengandung dua kelas.
    if y_proba is not None and np.unique(y_true_arr).size == 2:
        result["roc_auc"] = roc_auc_score(y_true_arr, y_proba)
    elif y_proba is not None:
        result["roc_auc"] = np.nan

    return result


def _mean_metric_dict(metric_rows: list[dict[str, float]]) -> dict[str, float]:
    """Rata-rata metrik per fold dengan dukungan NaN."""
    if not metric_rows:
        raise ValueError("Daftar metrik kosong.")
    keys = metric_rows[0].keys()
    return {
        key: float(np.nanmean([row[key] for row in metric_rows]))
        for key in keys
    }


def _print_formatted_results(results: dict) -> None:
    """Mencetak ringkasan hasil 5-fold CV."""
    col_w = [14, 12, 10, 11, 8, 10, 8, 8, 9]
    headers = [
        "Model", "Skenario", "Accuracy", "Precision",
        "Recall", "F1-Score", "FPR", "FNR", "ROC-AUC"
    ]
    sep = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"
    header_row = "| " + " | ".join(
        h.ljust(col_w[i]) for i, h in enumerate(headers)
    ) + " |"

    width = sum(col_w) + len(col_w) * 3
    print("\n" + "=" * width)
    print(f"{'TABEL MATRIKS RATA-RATA EVALUASI 5-FOLD CROSS VALIDATION':^{width}}")
    print("=" * width)
    print(sep)
    print(header_row)
    print(sep)

    for model_name, scenarios in results.items():
        for sc_name, metrics in scenarios.items():
            auc = metrics.get("roc_auc", np.nan)
            auc_text = f"{auc:.4f}" if np.isfinite(auc) else "NaN"
            row = "| " + " | ".join([
                model_name.ljust(col_w[0]),
                sc_name.ljust(col_w[1]),
                f"{metrics['accuracy']:.4f}".ljust(col_w[2]),
                f"{metrics['precision']:.4f}".ljust(col_w[3]),
                f"{metrics['recall']:.4f}".ljust(col_w[4]),
                f"{metrics['f1']:.4f}".ljust(col_w[5]),
                f"{metrics['fpr']:.4f}".ljust(col_w[6]),
                f"{metrics['fnr']:.4f}".ljust(col_w[7]),
                auc_text.ljust(col_w[8]),
            ]) + " |"
            print(row)
        print(sep)


def _export_metrics_to_csv(results: dict, timestamp: str) -> Path:
    """Mengekspor rata-rata metrik 5-fold CV."""
    records = []
    for model_name, scenarios in results.items():
        for sc_name, metrics in scenarios.items():
            records.append({
                "Model": model_name,
                "Skenario": sc_name,
                "Accuracy": metrics["accuracy"],
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "F1-Score": metrics["f1"],
                "FPR": metrics["fpr"],
                "FNR": metrics["fnr"],
                "ROC-AUC": metrics.get("roc_auc", np.nan),
            })

    out_csv = METRIC_OUT_DIR / f"metrik_kfold_{timestamp}.csv"
    pd.DataFrame(records).to_csv(out_csv, index=False)
    print(f"\n✅ Hasil 5-Fold CV tersimpan di: {out_csv}")
    return out_csv


def _plot_learning_curve(
    estimator: Any,
    title: str,
    X: pd.DataFrame,
    y: pd.Series,
    ax,
) -> None:
    """Membuat learning curve sederhana."""
    holdout_split = ShuffleSplit(
        n_splits=1, test_size=0.176, random_state=RANDOM_STATE
    )

    train_sizes, train_scores, val_scores = learning_curve(
        estimator,
        X,
        y,
        cv=holdout_split,
        n_jobs=1,
        train_sizes=np.linspace(0.1, 1.0, 5),
        scoring="accuracy",
    )

    ax.plot(train_sizes, np.mean(train_scores, axis=1), "o-", label="Skor Latih")
    ax.plot(train_sizes, np.mean(val_scores, axis=1), "o-", label="Skor Validasi")
    ax.set_title(title)
    ax.set_xlabel("Jumlah Data Latih")
    ax.set_ylabel("Akurasi")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)


def load_training_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Memuat baseline + EMPAT file obfuskasi yang sama dengan pipeline 5.400 baris.
    Tidak lagi menggunakan fitur_train_set_obf.csv 1.500 baris.
    """
    if not BASELINE_CSV.exists():
        raise FileNotFoundError(
            f"Berkas baseline tidak ditemukan: {BASELINE_CSV.resolve()}"
        )

    df_base = pd.read_csv(BASELINE_CSV).fillna(0)
    _validate_required_columns(df_base, BASELINE_CSV.name)

    list_df_aug = []
    print("[*] Memuat 4 file train set obfuskasi:")
    for file_name in AUGMENTATION_FILES:
        file_path = INPUT_DIR / file_name
        if not file_path.exists():
            raise FileNotFoundError(
                f"Berkas augmentasi tidak ditemukan: {file_path.resolve()}"
            )
        part = pd.read_csv(file_path).fillna(0)
        _validate_required_columns(part, file_name)
        list_df_aug.append(part)
        print(f"    - {file_name}: {len(part):,} baris")

    df_aug = pd.concat(list_df_aug, ignore_index=True)

    feature_cols = _get_feature_cols(df_base)
    missing_features = [c for c in feature_cols if c not in df_aug.columns]
    if missing_features:
        raise KeyError(
            "Kolom fitur baseline yang tidak ditemukan pada data augmentasi: "
            f"{missing_features[:20]}"
        )

    # Pastikan angka data benar-benar sama dengan eksperimen acuan.
    if len(df_base) != EXPECTED_BASELINE_ROWS:
        raise ValueError(
            f"Baseline harus {EXPECTED_BASELINE_ROWS:,} baris, "
            f"tetapi terbaca {len(df_base):,}."
        )
    if len(df_aug) != EXPECTED_AUGMENTATION_ROWS:
        raise ValueError(
            f"Train set obfuskasi harus {EXPECTED_AUGMENTATION_ROWS:,} baris, "
            f"tetapi terbaca {len(df_aug):,}."
        )

    print("\n📊 Dataset yang digunakan:")
    print(f"   Baseline murni       : {len(df_base):,} baris")
    print(f"   Train set obfuskasi  : {len(df_aug):,} baris")
    return df_base, df_aug


def _build_fold_data(
    df_base: pd.DataFrame,
    df_aug: pd.DataFrame,
):
    """
    Generator fold tunggal yang dipakai BERSAMA oleh CV utama dan ablation.
    Dengan ini train/validation split dan filter anti-leakage identik.
    """
    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(df_base, df_base[LABEL_COL]),
        start=1,
    ):
        df_fold_train = df_base.iloc[train_idx].copy()
        df_fold_val = df_base.iloc[val_idx].copy()

        # Hanya source malicious validasi yang memiliki turunan obfuskasi.
        val_mal_fnames = set(
            df_fold_val.loc[
                df_fold_val[LABEL_COL] == 1, FILENAME_COL
            ].astype(str)
        )

        # Anti-leakage: semua turunan source validasi dikeluarkan dari train augmentasi.
        aug_filename_as_str = df_aug[FILENAME_COL].astype(str)
        mask_val_derivative = aug_filename_as_str.isin(val_mal_fnames)

        df_aug_safe = df_aug.loc[~mask_val_derivative].copy()
        df_aug_heldout = df_aug.loc[mask_val_derivative].copy()

        df_train_aug = pd.concat(
            [df_fold_train, df_aug_safe],
            ignore_index=True,
        ).sample(
            frac=1.0,
            random_state=RANDOM_STATE,
        ).reset_index(drop=True)

        meta = {
            "fold": fold_idx,
            "n_val": len(df_fold_val),
            "n_val_mal": int((df_fold_val[LABEL_COL] == 1).sum()),
            "n_aug_before": len(df_aug),
            "n_aug_safe": len(df_aug_safe),
            "n_aug_removed": len(df_aug_heldout),
        }

        yield (
            fold_idx,
            df_fold_train,
            df_fold_val,
            df_aug_safe,
            df_aug_heldout,
            df_train_aug,
            meta,
        )


def verify_reference_folds(
    df_base: pd.DataFrame,
    df_aug: pd.DataFrame,
) -> list[dict[str, int]]:
    """Memastikan fold sama dengan output referensi yang diberikan."""
    observed = []

    for (
        fold_idx,
        _df_train,
        _df_val,
        _df_aug_safe,
        _df_aug_heldout,
        _df_train_aug,
        meta,
    ) in _build_fold_data(df_base, df_aug):
        observed.append(meta)
        expected = REFERENCE_FOLD_PROFILE[fold_idx - 1]
        actual = (
            meta["n_val"],
            meta["n_val_mal"],
            meta["n_aug_safe"],
            meta["n_aug_removed"],
        )
        if actual != expected:
            raise ValueError(
                f"Fold {fold_idx} tidak sama dengan referensi.\n"
                f"  Expected: validasi={expected[0]}, malicious={expected[1]}, "
                f"aug_safe={expected[2]}, dihapus={expected[3]}\n"
                f"  Actual  : validasi={actual[0]}, malicious={actual[1]}, "
                f"aug_safe={actual[2]}, dihapus={actual[3]}\n"
                "Periksa urutan/isi fitur_train_set.csv, empat file obfuskasi, "
                "RANDOM_STATE, atau versi dataset."
            )

    print("✅ Seluruh profil Fold 1–5 identik dengan output referensi.")
    return observed


def run_kfold_cross_validation(
    df_base: pd.DataFrame,
    df_aug: pd.DataFrame,
) -> tuple[dict, dict, list[dict[str, int]]]:
    """5-fold CV Baseline vs Augmentasi menggunakan fold yang sama."""
    print("\n[*] Menjalankan 5-Fold Cross Validation Pipeline...")
    print("    PROTEKSI LEAKAGE: turunan obfuskasi source validasi dikeluarkan dari train")

    feature_cols = _get_feature_cols(df_base)

    fold_metrics = {
        model_name: {"Baseline": [], "Augmentasi": []}
        for model_name in MODELS
    }
    fold_meta = []

    for (
        fold_idx,
        df_fold_train,
        df_fold_val,
        _df_aug_safe,
        _df_aug_heldout,
        df_fold_train_aug,
        meta,
    ) in _build_fold_data(df_base, df_aug):

        fold_meta.append(meta)

        print(f"\n  ── Fold {fold_idx}/{N_SPLITS} " + "─" * 39)
        print(
            f"     Validasi: {meta['n_val']} skrip "
            f"({meta['n_val_mal']} malicious)"
        )
        print(
            f"     Augmentasi: {meta['n_aug_before']} → {meta['n_aug_safe']} "
            f"(dihapus {meta['n_aug_removed']} turunan validasi)"
        )

        X_train_base = df_fold_train[feature_cols]
        y_train_base = df_fold_train[LABEL_COL]
        X_train_aug = df_fold_train_aug[feature_cols]
        y_train_aug = df_fold_train_aug[LABEL_COL]
        X_val = df_fold_val[feature_cols]
        y_val = df_fold_val[LABEL_COL]

        for model_name, model_template in MODELS.items():
            model_base = clone(model_template)
            model_base.fit(X_train_base, y_train_base)

            pred_base = model_base.predict(X_val)
            proba_base = model_base.predict_proba(X_val)[:, 1]
            metrics_base = _evaluate_metrics(y_val, pred_base, proba_base)

            model_aug = clone(model_template)
            model_aug.fit(X_train_aug, y_train_aug)

            pred_aug = model_aug.predict(X_val)
            proba_aug = model_aug.predict_proba(X_val)[:, 1]
            metrics_aug = _evaluate_metrics(y_val, pred_aug, proba_aug)

            fold_metrics[model_name]["Baseline"].append(metrics_base)
            fold_metrics[model_name]["Augmentasi"].append(metrics_aug)

            print(
                f"     {model_name:<14} "
                f"F1 Base={metrics_base['f1']:.4f} | "
                f"F1 Aug={metrics_aug['f1']:.4f}"
            )

    summary_results = {
        model_name: {
            scenario: _mean_metric_dict(metric_rows)
            for scenario, metric_rows in scenarios.items()
        }
        for model_name, scenarios in fold_metrics.items()
    }

    return summary_results, fold_metrics, fold_meta


def fit_and_export_final_models(
    df_base: pd.DataFrame,
    df_aug: pd.DataFrame,
) -> None:
    """Retrain penuh model baseline dan augmentasi lalu simpan artifact."""
    print("\n" + "=" * 80)
    print(" SIKLUS RETRAIN PENUH DAN EKSPOR MODEL")
    print("=" * 80)

    feature_cols = _get_feature_cols(df_base)

    df_full_aug = pd.concat(
        [df_base, df_aug],
        ignore_index=True,
    ).sample(
        frac=1.0,
        random_state=RANDOM_STATE,
    ).reset_index(drop=True)

    X_base = df_base[feature_cols]
    y_base = df_base[LABEL_COL]
    X_aug = df_full_aug[feature_cols]
    y_aug = df_full_aug[LABEL_COL]

    for model_name, model_template in MODELS.items():
        print(f"\n[+] Retrain final: {model_name}")

        model_base = clone(model_template)
        model_base.fit(X_base, y_base)

        model_aug = clone(model_template)
        model_aug.fit(X_aug, y_aug)

        base_path = (
            MODEL_OUT_DIR /
            f"model_{model_name}_Baseline_{waktu_sekarang}.joblib"
        )
        aug_path = (
            MODEL_OUT_DIR /
            f"model_{model_name}_Augmentasi_{waktu_sekarang}.joblib"
        )

        joblib.dump(model_base, base_path)
        joblib.dump(model_aug, aug_path)

        print(f"    Model Baseline   : {base_path}")
        print(f"    Model Augmentasi : {aug_path}")

        # Learning curve model augmentasi.
        fig, ax = plt.subplots(figsize=(8, 5))
        _plot_learning_curve(
            clone(model_template),
            f"Learning Curve — {model_name} Augmentasi",
            X_aug,
            y_aug,
            ax,
        )
        fig.tight_layout()
        lc_path = (
            GRAPH_OUT_DIR /
            f"learning_curve_{model_name}_{waktu_sekarang}.png"
        )
        fig.savefig(lc_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        # Feature importance bila model mendukungnya.
        if hasattr(model_aug, "feature_importances_"):
            importance = pd.Series(
                model_aug.feature_importances_,
                index=feature_cols,
            ).sort_values(ascending=False).head(25)

            fig2, ax2 = plt.subplots(figsize=(9, 7))
            importance.sort_values().plot(kind="barh", ax=ax2)
            ax2.set_title(f"Top 25 Feature Importance — {model_name}")
            ax2.set_xlabel("Importance")
            fig2.tight_layout()
            fi_path = (
                GRAPH_OUT_DIR /
                f"feature_importance_{model_name}_{waktu_sekarang}.png"
            )
            fig2.savefig(fi_path, dpi=300, bbox_inches="tight")
            plt.close(fig2)

    print("\n✅ Retrain dan ekspor model selesai.")


print("✅ [CELL 1] Seluruh fungsi pembantu dan pipeline telah didefinisikan.")


✅ [CELL 1] Seluruh fungsi pembantu dan pipeline telah didefinisikan.


In [3]:
# =====================================================================
# CELL 2 — LOAD DATA 4.949 BASELINE + 5.400 OBFUSKASI
#            DAN VERIFIKASI PROFIL FOLD REFERENSI
# =====================================================================

df_train_base, df_aug_5400 = load_training_data()

# Alias generik untuk dipakai sel berikutnya.
df_train_obf = df_aug_5400

# Verifikasi keras agar ablation/statistik tidak diam-diam memakai data berbeda.
reference_fold_metadata = verify_reference_folds(df_train_base, df_train_obf)

print("\nProfil fold terverifikasi:")
for m in reference_fold_metadata:
    print(
        f"  Fold {m['fold']}: validasi={m['n_val']} "
        f"({m['n_val_mal']} malicious), "
        f"augmentasi={m['n_aug_before']}→{m['n_aug_safe']} "
        f"(dihapus {m['n_aug_removed']})"
    )

print("\n✅ [CELL 2] Dataset 5.400 dan fold referensi siap dipakai bersama.")


[*] Memuat 4 file train set obfuskasi:
    - fitur_train_set_obf_ASCII_Encoding.csv: 1,350 baris
    - fitur_train_set_obf_String_Concatenation.csv: 1,350 baris
    - fitur_train_set_obf_String_Reordering.csv: 1,350 baris
    - fitur_train_set_obf_Token_Manipulation.csv: 1,350 baris

📊 Dataset yang digunakan:
   Baseline murni       : 4,949 baris
   Train set obfuskasi  : 5,400 baris
✅ Seluruh profil Fold 1–5 identik dengan output referensi.

Profil fold terverifikasi:
  Fold 1: validasi=990 (301 malicious), augmentasi=5400→4340 (dihapus 1060)
  Fold 2: validasi=990 (301 malicious), augmentasi=5400→4296 (dihapus 1104)
  Fold 3: validasi=990 (301 malicious), augmentasi=5400→4292 (dihapus 1108)
  Fold 4: validasi=990 (302 malicious), augmentasi=5400→4348 (dihapus 1052)
  Fold 5: validasi=989 (301 malicious), augmentasi=5400→4324 (dihapus 1076)

✅ [CELL 2] Dataset 5.400 dan fold referensi siap dipakai bersama.


In [4]:
# =====================================================================
# CELL 3 — PIPELINE UTAMA: BASELINE VS AUGMENTASI 5.400
# =====================================================================

print("=" * 80)
print(
    f"▶ MULAI EKSEKUSI PIPELINE MODELING: "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)
print("=" * 80)

waktu_mulai_modeling = time.time()

summary_results, cv_fold_metrics, cv_fold_metadata = run_kfold_cross_validation(
    df_train_base,
    df_train_obf,
)

_print_formatted_results(summary_results)
_export_metrics_to_csv(summary_results, waktu_sekarang)

# Simpan F1 per fold agar dapat diaudit.
fold_records = []
for model_name, scenarios in cv_fold_metrics.items():
    for scenario, rows in scenarios.items():
        for fold_idx, metrics in enumerate(rows, start=1):
            fold_records.append({
                "Model": model_name,
                "Skenario": scenario,
                "Fold": fold_idx,
                **{k: v for k, v in metrics.items()},
            })

df_cv_folds = pd.DataFrame(fold_records)
cv_fold_out = METRIC_OUT_DIR / f"metrik_kfold_per_fold_{waktu_sekarang}.csv"
df_cv_folds.to_csv(cv_fold_out, index=False)
print(f"✅ Metrik per fold tersimpan di: {cv_fold_out}")

# Retrain final model.
fit_and_export_final_models(df_train_base, df_train_obf)

durasi_modeling = (time.time() - waktu_mulai_modeling) / 60
print("=" * 80)
print(
    f"⏹ EKSEKUSI MODELING SELESAI. "
    f"Total Waktu: {durasi_modeling:.2f} Menit."
)
print("=" * 80)


▶ MULAI EKSEKUSI PIPELINE MODELING: 2026-08-09 10:13:14

[*] Menjalankan 5-Fold Cross Validation Pipeline...
    PROTEKSI LEAKAGE: turunan obfuskasi source validasi dikeluarkan dari train

  ── Fold 1/5 ───────────────────────────────────────
     Validasi: 990 skrip (301 malicious)
     Augmentasi: 5400 → 4340 (dihapus 1060 turunan validasi)
     RandomForest   F1 Base=0.9605 | F1 Aug=0.9513
     XGBoost        F1 Base=0.9525 | F1 Aug=0.9400
     LightGBM       F1 Base=0.9577 | F1 Aug=0.9445

  ── Fold 2/5 ───────────────────────────────────────
     Validasi: 990 skrip (301 malicious)
     Augmentasi: 5400 → 4296 (dihapus 1104 turunan validasi)
     RandomForest   F1 Base=0.9663 | F1 Aug=0.9601
     XGBoost        F1 Base=0.9700 | F1 Aug=0.9555
     LightGBM       F1 Base=0.9715 | F1 Aug=0.9572

  ── Fold 3/5 ───────────────────────────────────────
     Validasi: 990 skrip (301 malicious)
     Augmentasi: 5400 → 4292 (dihapus 1108 turunan validasi)
     RandomForest   F1 Base=0.9676 

In [5]:
# =====================================================================
# CELL 4 — ABLATION STUDY
#
# PENTING:
# - Train set obfuskasi memakai df_train_obf = 5.400 baris yang SAMA
#   dengan pipeline utama.
# - Split fold memakai _build_fold_data() yang SAMA dengan pipeline utama.
# - Anti-leakage identik: turunan source malicious validasi dikeluarkan
#   dari training dan dipakai sebagai held-out obfuscation validation.
# - Hasil disimpan per model, per kelompok fitur, per fold.
# =====================================================================

print("=" * 80)
print("ABLATION STUDY — TRAIN OBFUSKASI 5.400 + ANTI-LEAKAGE")
print("=" * 80)

t_start_abl = time.time()

feature_cols_all = _get_feature_cols(df_train_base)

feat_ast = [c for c in feature_cols_all if c.endswith("Ast")]
feat_lex = [c for c in feature_cols_all if c.startswith("tok_")]
feat_stat = [
    c for c in feature_cols_all
    if c not in feat_ast and c not in feat_lex
]

ablation_groups = {
    "Statistik Saja": feat_stat,
    "Leksikal Saja": feat_lex,
    "AST Saja": feat_ast,
    "Statistik+Leksikal": feat_stat + feat_lex,
    "Statistik+AST": feat_stat + feat_ast,
    "Leksikal+AST": feat_lex + feat_ast,
    "Hybrid (Semua Fitur)": feature_cols_all,
}

# Validasi agar tidak ada grup fitur kosong.
empty_groups = [name for name, cols in ablation_groups.items() if not cols]
if empty_groups:
    raise ValueError(
        f"Kelompok fitur berikut kosong: {empty_groups}. "
        "Periksa pola nama kolom tok_ dan *Ast."
    )

print("\nJumlah fitur:")
for group_name, cols in ablation_groups.items():
    print(f"  {group_name:<24}: {len(cols)}")

# Struktur ini sengaja kompatibel dengan CELL CI:
# ablation_results[model][group]["Murni"] -> list 5 F1
# ablation_results[model][group]["Obfuskasi"] -> list 5 F1
ablation_results = {
    model_name: {
        group_name: {"Murni": [], "Obfuskasi": []}
        for group_name in ablation_groups
    }
    for model_name in MODELS
}

ablation_records = []

for (
    fold_idx,
    _df_fold_train,
    df_fold_val_murni,
    _df_aug_safe,
    df_fold_val_obf,
    df_fold_train_aug,
    meta,
) in _build_fold_data(df_train_base, df_train_obf):

    print(f"\n  ── Fold {fold_idx}/{N_SPLITS} " + "─" * 39)
    print(
        f"     Validasi murni : {meta['n_val']} skrip "
        f"({meta['n_val_mal']} malicious)"
    )
    print(
        f"     Train obf aman : {meta['n_aug_before']} → {meta['n_aug_safe']} "
        f"(dihapus {meta['n_aug_removed']} turunan validasi)"
    )
    print(f"     Validasi obf   : {len(df_fold_val_obf)} turunan held-out")

    if df_fold_val_obf.empty:
        raise ValueError(
            f"Fold {fold_idx} tidak memiliki held-out obfuscated validation."
        )

    for model_name, model_template in MODELS.items():
        for group_name, features in ablation_groups.items():
            # Pastikan semua fitur tersedia di ketiga dataframe.
            missing_train = [
                c for c in features if c not in df_fold_train_aug.columns
            ]
            missing_val = [
                c for c in features if c not in df_fold_val_murni.columns
            ]
            missing_obf = [
                c for c in features if c not in df_fold_val_obf.columns
            ]
            if missing_train or missing_val or missing_obf:
                raise KeyError(
                    f"Kolom fitur hilang pada {model_name}/{group_name}/Fold {fold_idx}. "
                    f"train={missing_train[:5]}, val={missing_val[:5]}, "
                    f"obf={missing_obf[:5]}"
                )

            X_train = df_fold_train_aug[features]
            y_train = df_fold_train_aug[LABEL_COL]

            X_val_murni = df_fold_val_murni[features]
            y_val_murni = df_fold_val_murni[LABEL_COL]

            X_val_obf = df_fold_val_obf[features]
            y_val_obf = df_fold_val_obf[LABEL_COL]

            model = clone(model_template)
            model.fit(X_train, y_train)

            pred_murni = model.predict(X_val_murni)
            pred_obf = model.predict(X_val_obf)

            # F1 murni: validasi normal dua kelas.
            f1_murni = f1_score(
                y_val_murni, pred_murni, zero_division=0
            )

            # F1 obf: held-out turunan obfuskasi dari source malicious validasi.
            # Dipakai langsung agar tidak memanggil ROC-AUC pada one-class set.
            f1_obf = f1_score(
                y_val_obf, pred_obf, zero_division=0
            )

            ablation_results[model_name][group_name]["Murni"].append(
                float(f1_murni)
            )
            ablation_results[model_name][group_name]["Obfuskasi"].append(
                float(f1_obf)
            )

            ablation_records.append({
                "Fold": fold_idx,
                "Model": model_name,
                "Kelompok_Fitur": group_name,
                "Jumlah_Fitur": len(features),
                "F1_Murni": float(f1_murni),
                "F1_Obfuskasi": float(f1_obf),
                "F1_Combined": float((f1_murni + f1_obf) / 2),
                "Train_Obf_Sebelum": meta["n_aug_before"],
                "Train_Obf_Aman": meta["n_aug_safe"],
                "Turunan_Validasi_Dihapus": meta["n_aug_removed"],
            })

# ---------------------------------------------------------------------
# Ringkasan ablation
# ---------------------------------------------------------------------
print("\n" + "=" * 96)
print("RINGKASAN ABLATION STUDY — RATA-RATA 5 FOLD")
print("=" * 96)

for model_name in MODELS:
    print(f"\n>>> {model_name}")
    print(
        f"{'Kelompok Fitur':<24} "
        f"{'F1 Murni':>12} "
        f"{'F1 Obf':>12} "
        f"{'Combined':>12}"
    )
    print("-" * 64)

    for group_name in ablation_groups:
        murni_scores = np.array(
            ablation_results[model_name][group_name]["Murni"],
            dtype=float,
        )
        obf_scores = np.array(
            ablation_results[model_name][group_name]["Obfuskasi"],
            dtype=float,
        )
        combined_scores = (murni_scores + obf_scores) / 2

        print(
            f"{group_name:<24} "
            f"{murni_scores.mean():>12.4f} "
            f"{obf_scores.mean():>12.4f} "
            f"{combined_scores.mean():>12.4f}"
        )

# Simpan detail per fold.
df_ablation_folds = pd.DataFrame(ablation_records)
ablation_fold_out = (
    METRIC_OUT_DIR /
    f"ablation_per_fold_5400_{waktu_sekarang}.csv"
)
df_ablation_folds.to_csv(ablation_fold_out, index=False)

# Simpan ringkasan.
summary_abl_records = []
for model_name in MODELS:
    for group_name, features in ablation_groups.items():
        murni = np.array(
            ablation_results[model_name][group_name]["Murni"],
            dtype=float,
        )
        obf = np.array(
            ablation_results[model_name][group_name]["Obfuskasi"],
            dtype=float,
        )
        combined = (murni + obf) / 2

        summary_abl_records.append({
            "Model": model_name,
            "Kelompok_Fitur": group_name,
            "Jumlah_Fitur": len(features),
            "Mean_F1_Murni": murni.mean(),
            "Mean_F1_Obfuskasi": obf.mean(),
            "Mean_F1_Combined": combined.mean(),
        })

df_ablation_summary = pd.DataFrame(summary_abl_records)
ablation_summary_out = (
    METRIC_OUT_DIR /
    f"ablation_summary_5400_{waktu_sekarang}.csv"
)
df_ablation_summary.to_csv(ablation_summary_out, index=False)

print(f"\n✅ Detail ablation tersimpan : {ablation_fold_out}")
print(f"✅ Ringkasan ablation        : {ablation_summary_out}")
print(
    f"✅ Ablation selesai dalam "
    f"{(time.time() - t_start_abl) / 60:.2f} menit."
)


ABLATION STUDY — TRAIN OBFUSKASI 5.400 + ANTI-LEAKAGE

Jumlah fitur:
  Statistik Saja          : 4
  Leksikal Saja           : 111
  AST Saja                : 12
  Statistik+Leksikal      : 115
  Statistik+AST           : 16
  Leksikal+AST            : 123
  Hybrid (Semua Fitur)    : 127

  ── Fold 1/5 ───────────────────────────────────────
     Validasi murni : 990 skrip (301 malicious)
     Train obf aman : 5400 → 4340 (dihapus 1060 turunan validasi)
     Validasi obf   : 1060 turunan held-out

  ── Fold 2/5 ───────────────────────────────────────
     Validasi murni : 990 skrip (301 malicious)
     Train obf aman : 5400 → 4296 (dihapus 1104 turunan validasi)
     Validasi obf   : 1104 turunan held-out

  ── Fold 3/5 ───────────────────────────────────────
     Validasi murni : 990 skrip (301 malicious)
     Train obf aman : 5400 → 4292 (dihapus 1108 turunan validasi)
     Validasi obf   : 1108 turunan held-out

  ── Fold 4/5 ───────────────────────────────────────
     Validasi mur

In [6]:
# =====================================================================
# CELL 5 — CONFIDENCE INTERVAL & UJI STATISTIK ANTAR MODEL
#
# Menghitung 95% CI dari distribusi F1-Score per fold (5 nilai)
# dan uji Wilcoxon signed-rank antar model.
#
# - CI: t-distribution, df=4 karena n=5 fold
# - Wilcoxon: non-parametrik, tidak mengasumsikan normalitas
#
# Data statistik diambil LANGSUNG dari ablation_results CELL 4,
# sehingga train obfuskasi tetap memakai 5.400 data yang sama.
# =====================================================================

print("=" * 80)
print(" CONFIDENCE INTERVAL (95%) DAN UJI STATISTIK ANTAR MODEL")
print("=" * 80)

TARGET_GROUP = "Hybrid (Semua Fitur)"

if "ablation_results" not in globals():
    raise NameError(
        "ablation_results belum tersedia. Jalankan CELL 4 terlebih dahulu."
    )

if TARGET_GROUP not in ablation_groups:
    raise KeyError(
        f"TARGET_GROUP '{TARGET_GROUP}' tidak ditemukan dalam ablation_groups."
    )

# ---------------------------------------------------------------------
# 1. Ambil skor F1 per fold dari grup Hybrid
# ---------------------------------------------------------------------
fold_scores = {}

for model_name in MODELS:
    murni_scores = ablation_results[model_name][TARGET_GROUP]["Murni"]
    obf_scores = ablation_results[model_name][TARGET_GROUP]["Obfuskasi"]

    if len(murni_scores) != N_SPLITS or len(obf_scores) != N_SPLITS:
        raise ValueError(
            f"{model_name}: skor harus berjumlah {N_SPLITS} fold, "
            f"tetapi Murni={len(murni_scores)}, "
            f"Obfuskasi={len(obf_scores)}."
        )

    # Sesuai rancangan yang Anda berikan:
    # F1 combined per fold = rata-rata F1 murni dan F1 obfuskasi.
    combined = [
        (float(m) + float(o)) / 2.0
        for m, o in zip(murni_scores, obf_scores)
    ]
    fold_scores[model_name] = combined

# Audit skor yang benar-benar diuji.
print("\nSkor F1 combined per fold:")
for model_name, scores in fold_scores.items():
    print(
        f"  {model_name:<14}: "
        + ", ".join(f"{s:.4f}" for s in scores)
    )

# ---------------------------------------------------------------------
# 2. Hitung 95% Confidence Interval per model
# ---------------------------------------------------------------------
print(
    f"\n{'Model':<30} "
    f"{'Mean F1':>10} "
    f"{'CI Lower':>10} "
    f"{'CI Upper':>10} "
    f"{'±':>10}"
)
print("-" * 75)

ci_results = {}

for model_name, scores in fold_scores.items():
    arr = np.asarray(scores, dtype=float)

    if arr.size < 2:
        raise ValueError(
            f"{model_name}: minimal 2 skor diperlukan untuk menghitung CI."
        )

    mean = float(arr.mean())
    sem = float(stats.sem(arr))

    # Jika seluruh nilai identik, SEM=0 dan CI tepat di mean.
    if np.isclose(sem, 0.0):
        ci_lower = mean
        ci_upper = mean
    else:
        ci_lower, ci_upper = stats.t.interval(
            0.95,
            df=arr.size - 1,
            loc=mean,
            scale=sem,
        )
        ci_lower = float(ci_lower)
        ci_upper = float(ci_upper)

    margin = float(ci_upper - mean)

    ci_results[model_name] = {
        "mean": mean,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "margin": margin,
    }

    print(
        f"{model_name:<30} "
        f"{mean:>10.4f} "
        f"{ci_lower:>10.4f} "
        f"{ci_upper:>10.4f} "
        f"{margin:>10.4f}"
    )

print("-" * 75)
print("Interpretasi: Mean F1 ± margin pada tingkat kepercayaan 95%.")
print("Metode CI: Student's t-distribution, df=4 karena n=5 fold.")

# ---------------------------------------------------------------------
# 3. Uji Wilcoxon signed-rank antar pasangan model
# ---------------------------------------------------------------------
print("\n" + "=" * 80)
print(" UJI WILCOXON SIGNED-RANK")
print(" H0: distribusi selisih skor paired antar model berpusat di nol")
print("=" * 80)

print(
    f"\n{'Perbandingan':<45} "
    f"{'Statistic':>10} "
    f"{'p-value':>10} "
    f"{'Signifikan':>14}"
)
print("-" * 84)

wilcoxon_records = []
model_list = list(fold_scores.keys())

for i in range(len(model_list)):
    for j in range(i + 1, len(model_list)):
        m1 = model_list[i]
        m2 = model_list[j]

        s1 = np.asarray(fold_scores[m1], dtype=float)
        s2 = np.asarray(fold_scores[m2], dtype=float)
        diff = s1 - s2

        # scipy.stats.wilcoxon gagal bila seluruh paired difference = 0.
        if np.allclose(diff, 0.0):
            stat = 0.0
            p = 1.0
            note = "Semua selisih = 0"
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                stat, p = wilcoxon(
                    s1,
                    s2,
                    zero_method="wilcox",
                    alternative="two-sided",
                )
            stat = float(stat)
            p = float(p)
            note = ""

        significant = p < 0.05
        sig_text = "✓ Ya (p<0.05)" if significant else "✗ Tidak"
        label = f"{m1} vs {m2}"

        print(
            f"{label:<45} "
            f"{stat:>10.3f} "
            f"{p:>10.4f} "
            f"{sig_text:>14}"
        )

        wilcoxon_records.append({
            "Model_1": m1,
            "Model_2": m2,
            "Statistic": stat,
            "p_value": p,
            "Signifikan_0.05": significant,
            "Catatan": note,
        })

print("-" * 84)
print("Wilcoxon dipilih karena bersifat non-parametrik dan paired per fold.")
print("Kriteria keputusan: p < 0.05 → perbedaan signifikan secara statistik.")
print(
    "Catatan metodologis: dengan hanya 5 pasangan fold, daya uji Wilcoxon "
    "dua sisi sangat terbatas; hasil tidak signifikan harus dilaporkan apa adanya."
)

# ---------------------------------------------------------------------
# 4. Simpan CI dan Wilcoxon ke CSV
# ---------------------------------------------------------------------
ci_records = []
for model_name, ci in ci_results.items():
    ci_records.append({
        "Model": model_name,
        "Target_Group": TARGET_GROUP,
        "Mean_F1": round(ci["mean"], 6),
        "CI_Lower": round(ci["ci_lower"], 6),
        "CI_Upper": round(ci["ci_upper"], 6),
        "Margin_CI": round(ci["margin"], 6),
        "N_Fold": N_SPLITS,
    })

df_ci = pd.DataFrame(ci_records)
ci_out = METRIC_OUT_DIR / f"ci_statistik_{waktu_sekarang}.csv"
df_ci.to_csv(ci_out, index=False)

df_wilcoxon = pd.DataFrame(wilcoxon_records)
wilcoxon_out = METRIC_OUT_DIR / f"wilcoxon_antar_model_{waktu_sekarang}.csv"
df_wilcoxon.to_csv(wilcoxon_out, index=False)

# Simpan pula skor paired yang menjadi input statistik.
paired_records = []
for model_name, scores in fold_scores.items():
    for fold_idx, score in enumerate(scores, start=1):
        paired_records.append({
            "Model": model_name,
            "Fold": fold_idx,
            "Target_Group": TARGET_GROUP,
            "F1_Combined": score,
            "F1_Murni": ablation_results[model_name][TARGET_GROUP]["Murni"][fold_idx - 1],
            "F1_Obfuskasi": ablation_results[model_name][TARGET_GROUP]["Obfuskasi"][fold_idx - 1],
        })

df_paired = pd.DataFrame(paired_records)
paired_out = METRIC_OUT_DIR / f"skor_statistik_per_fold_{waktu_sekarang}.csv"
df_paired.to_csv(paired_out, index=False)

print(f"\n✅ Hasil CI tersimpan       : {ci_out}")
print(f"✅ Hasil Wilcoxon tersimpan : {wilcoxon_out}")
print(f"✅ Skor paired tersimpan    : {paired_out}")
print("=" * 80)


 CONFIDENCE INTERVAL (95%) DAN UJI STATISTIK ANTAR MODEL

Skor F1 combined per fold:
  RandomForest  : 0.9744, 0.9794, 0.9785, 0.9760, 0.9692
  XGBoost       : 0.9691, 0.9771, 0.9779, 0.9780, 0.9670
  LightGBM      : 0.9718, 0.9782, 0.9786, 0.9778, 0.9657

Model                             Mean F1   CI Lower   CI Upper          ±
---------------------------------------------------------------------------
RandomForest                       0.9755     0.9705     0.9805     0.0050
XGBoost                            0.9738     0.9672     0.9804     0.0066
LightGBM                           0.9744     0.9675     0.9814     0.0069
---------------------------------------------------------------------------
Interpretasi: Mean F1 ± margin pada tingkat kepercayaan 95%.
Metode CI: Student's t-distribution, df=4 karena n=5 fold.

 UJI WILCOXON SIGNED-RANK
 H0: distribusi selisih skor paired antar model berpusat di nol

Perbandingan                                   Statistic    p-value     Signifi